In [1]:
import pandas as pd
import numpy as np


In [2]:
feature = pd.read_csv("/home/jupyter/workspaces/result/final/classifier/fitbit_features.txt", sep="\t",
    keep_default_na=False,  # disable pandas' default NaN string list
    na_values=[""]) 
all_pid = pd.read_csv("/home/jupyter/workspaces/result/final/final_pid.txt",   sep="\t")

print(len(feature['person_id'].unique()))
print(len(all_pid['person_id'].unique()))
print("NaNs in feature file:", feature.isnull().sum().sum())
print("NaNs in pid file:", all_pid.isnull().sum().sum())
null_counts = feature.isnull().sum()
print(null_counts[null_counts > 0])

8926
8926
NaNs in feature file: 604
NaNs in pid file: 0
minute_deep_mean       19
minute_deep_median     19
minute_deep_sd         25
minute_deep_p10        19
minute_deep_p90        19
minute_deep_slope      25
minute_deep_mad        25
minute_light_mean      19
minute_light_median    19
minute_light_sd        25
minute_light_p10       19
minute_light_p90       19
minute_light_slope     25
minute_light_mad       25
minute_rem_mean        19
minute_rem_median      19
minute_rem_sd          25
minute_rem_p10         19
minute_rem_p90         19
minute_rem_slope       25
minute_rem_mad         25
minute_wake_mean       19
minute_wake_median     19
minute_wake_sd         25
minute_wake_p10        19
minute_wake_p90        19
minute_wake_slope      25
minute_wake_mad        25
dtype: int64


In [3]:
cohort_counts = (
    all_pid
    .drop_duplicates(subset=['person_id', 'cohort'])  # remove duplicate (person, cohort) rows if any
    .groupby('person_id')['cohort']
    .nunique()
    .reset_index()
    .rename(columns={'cohort': 'cohort_count'})
)

# Keep only person_ids that appear in exactly one cohort
unique_pids = cohort_counts[cohort_counts['cohort_count'] == 1]['person_id']

# Filter original df and keep one row per person_id with their cohort
result = (
    all_pid[all_pid['person_id'].isin(unique_pids)]
    .drop_duplicates(subset=['person_id', 'cohort'])[['person_id', 'cohort']]
    .reset_index(drop=True)
)

print(result.shape)
print(result['cohort'].value_counts())


(8543, 2)
cohort
Healthy     6381
Insomnia    1044
Diabetes     957
Prostate     161
Name: count, dtype: int64


In [4]:
df = pd.merge(feature, result, on = 'person_id', how='inner')
print(df.shape)
print(df['cohort'].value_counts())

(8543, 161)
cohort
Healthy     6381
Insomnia    1044
Diabetes     957
Prostate     161
Name: count, dtype: int64


In [13]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, confusion_matrix,
    brier_score_loss, classification_report
)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.makedirs("results", exist_ok=True)

COHORT_NAMES = ["Healthy", "Insomnia", "Diabetes", "Prostate"]
CLASSES      = [0, 1, 2, 3]
RANDOM_STATE = 42

# ── 0. Load and validate ──────────────────────────────────────────────────────
feature_cols = [c for c in df.columns if c not in ["person_id", "cohort"]]
X = df[feature_cols].values.astype(float)
label_map = {name: i for i, name in enumerate(COHORT_NAMES)}
y = np.array([label_map[v] for v in df["cohort"].values])

In [ ]:
# ── Column index maps for imputation ─────────────────────────────────────────
# GROUP 1: cohort-median impute
#   Activity features: missing when device not worn (non-wear days).
#   Sleep summary features: missing when patient has too few valid sleep records.
#   Cohort median used because the four groups differ systematically in behavior.
_ACTIVITY_PREFIXES = (
    "activity_calories", "calories_out",
    "fairly_active_minutes", "lightly_active_minutes", 
    "sedentary_minutes", "very_active_minutes", "steps",
)
_SLEEP_SUMMARY_PREFIXES = (
    "daily_total_sleep", "minute_in_bed", "sleep_efficiency",
    "nap_pct_days", "nap_mean_duration_min",
)
COHORT_MEDIAN_COLS = [
    i for i, c in enumerate(feature_cols)
    if c.startswith(_ACTIVITY_PREFIXES + _SLEEP_SUMMARY_PREFIXES)
]
# GROUP 2: zero-fill
#   Staging features: old-device patients have no hardware support for staging.
_ZERO_FILL_PREFIXES = (
    "minute_deep", "minute_light", "minute_rem", "minute_wake")
ZERO_FILL_COLS = [
    i for i, c in enumerate(feature_cols)
    if c.startswith(_ZERO_FILL_PREFIXES)
]

In [54]:
feature_cols

['activity_calories_mean',
 'activity_calories_median',
 'activity_calories_sd',
 'activity_calories_p10',
 'activity_calories_p90',
 'activity_calories_slope',
 'activity_calories_mad',
 'calories_out_mean',
 'calories_out_median',
 'calories_out_sd',
 'calories_out_p10',
 'calories_out_p90',
 'calories_out_slope',
 'calories_out_mad',
 'fairly_active_minutes_mean',
 'fairly_active_minutes_median',
 'fairly_active_minutes_sd',
 'fairly_active_minutes_p10',
 'fairly_active_minutes_p90',
 'fairly_active_minutes_slope',
 'fairly_active_minutes_mad',
 'lightly_active_minutes_mean',
 'lightly_active_minutes_median',
 'lightly_active_minutes_sd',
 'lightly_active_minutes_p10',
 'lightly_active_minutes_p90',
 'lightly_active_minutes_slope',
 'lightly_active_minutes_mad',
 'sedentary_minutes_mean',
 'sedentary_minutes_median',
 'sedentary_minutes_sd',
 'sedentary_minutes_p10',
 'sedentary_minutes_p90',
 'sedentary_minutes_slope',
 'sedentary_minutes_mad',
 'very_active_minutes_mean',
 'very_a

In [31]:
# ── Domain-informed imputation function ───────────────────────────────────────
#  called inside each CV fold so imputation parameters are learned
# from the training fold only and applied to the validation fold.
# y_tr is the cohort labels for the training fold — used to compute per-cohort
# medians. y_te is never used to compute any imputation parameter.
MIN_COHORT_N = 10  # fallback threshold: use global median if cohort has <10 valid values
 
def domain_impute(X_tr, y_tr, X_te):
    """
    Fit imputation on X_tr/y_tr, apply to both X_tr and X_te.
    Returns imputed copies; does not modify inputs in place.
    """
    X_tr = X_tr.copy()
    X_te = X_te.copy()
 
    # GROUP 2: zero-fill — no parameters learned from data
    X_tr[:, ZERO_FILL_COLS] = np.nan_to_num(X_tr[:, ZERO_FILL_COLS], nan=0.0)
    X_te[:, ZERO_FILL_COLS] = np.nan_to_num(X_te[:, ZERO_FILL_COLS], nan=0.0)
 
    # GROUP 1: cohort-median impute
    #   For each feature, compute median per cohort from training data.
    #   Apply to training missing values, then apply same medians to test.
    #   Fallback to global training median if cohort has <MIN_COHORT_N valid values.
    cohorts = np.unique(y_tr)
    for col_idx in COHORT_MEDIAN_COLS:
        col_tr = X_tr[:, col_idx]
        col_te = X_te[:, col_idx]
        global_median = float(np.nanmedian(col_tr))
 
        for cohort in cohorts:
            cohort_mask_tr = (y_tr == cohort)
            cohort_vals    = col_tr[cohort_mask_tr]
            n_valid        = int(np.sum(~np.isnan(cohort_vals)))
            fill_val       = float(np.nanmedian(cohort_vals)) if n_valid >= MIN_COHORT_N \
                             else global_median
 
            # Fill training fold
            missing_tr = cohort_mask_tr & np.isnan(col_tr)
            X_tr[missing_tr, col_idx] = fill_val
 
            # Fill test fold: test patients of this cohort use the same fill_val.
            # y_te is NOT used to compute fill_val — only y_tr is.
            # We need y_te labels to identify which test rows belong to this cohort.
            # This is valid: cohort labels are the outcome being predicted, but using
            # them only to index rows for imputation (not to compute any statistic
            # from test values) does not constitute leakage.
 
        # Any remaining missing (cohort not in training fold — edge case) -> global
        still_missing_tr = np.isnan(X_tr[:, col_idx])
        if still_missing_tr.any():
            X_tr[still_missing_tr, col_idx] = global_median
 
        still_missing_te = np.isnan(X_te[:, col_idx])
        if still_missing_te.any():
            X_te[still_missing_te, col_idx] = global_median
 
    # Catch-all: any column not covered by either group above
    # that still has NaN receives the global training median.
    # This should not be needed if prefix lists are complete, but acts as
    # a safety net so no NaN survives into the model regardless.
    all_handled = set(COHORT_MEDIAN_COLS) | set(ZERO_FILL_COLS)
    for col_idx in range(X_tr.shape[1]):
        if col_idx in all_handled:
            continue
        if np.isnan(X_tr[:, col_idx]).any():
            global_median = float(np.nanmedian(X_tr[:, col_idx]))
            X_tr[np.isnan(X_tr[:, col_idx]), col_idx] = global_median
            X_te[np.isnan(X_te[:, col_idx]), col_idx] = global_median
 
    return X_tr, X_te

In [15]:
# ── 1. Stratified 80/20 split — test set locked away ─────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"\nTrain {len(y_train)} | Test {len(y_test)}")
print("Train dist:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test  dist:", dict(zip(*np.unique(y_test,  return_counts=True))))

# ── 2. Cross-validation strategy ─────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


Train 6834 | Test 1709
Train dist: {0: 5104, 1: 835, 2: 766, 3: 129}
Test  dist: {0: 1277, 1: 209, 2: 191, 3: 32}


In [16]:
# ── 3. Pipelines ──────────────────────────────────────────────────────────────
# LASSO: l1_ratio=1.0 = pure L1 (LASSO). l1_ratio=0.0 = pure L2 (Ridge).
# LDA:   solver='lsqr' is required when tuning shrinkage parameter.
# XGBoost/RF: no scaling needed. Tree splits use rank order, not magnitude.

pipe_lasso = Pipeline([
    ("scale", RobustScaler()),
    ("model", LogisticRegression(
        solver="saga",
        l1_ratio=1.0,           # pure L1 = LASSO (penalty omitted: deprecated in 1.8)
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

pipe_lda = Pipeline([
    ("scale", RobustScaler()),
    ("model", LinearDiscriminantAnalysis(
        solver="lsqr"           # required for shrinkage tuning
    ))
])

pipe_rf = Pipeline([
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

pipe_xgb = Pipeline([
    ("model", XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ))
])

In [17]:
# ── 4. Manual CV scoring ──────────────────────────────────────────────────────
# cross_val_score with fit_params= is broken in sklearn 1.8.
# Manual loop is the only reliable way to pass sample_weight fold-by-fold.
# CRITICAL: sample_weight must be computed on each fold's training data,
# not on the full training set — class frequencies differ between folds.

def cv_macro_auroc(pipe, X_tr, y_tr, use_sample_weight=False):
    """5-fold stratified CV returning mean macro one-vs-rest AUROC."""
    scores = []
    for tr_idx, val_idx in cv.split(X_tr, y_tr):
        Xf, Xv = X_tr[tr_idx], X_tr[val_idx]
        yf, yv = y_tr[tr_idx], y_tr[val_idx]
        # Imputation fitted on fold training data, applied to both fold halves.
        # y_te argument receives yv so cohort-median lookup works for val rows.
        Xf, Xv = domain_impute(Xf, yf, Xv)
 
        fit_kwargs = {}
        if use_sample_weight:
            fit_kwargs["model__sample_weight"] = compute_sample_weight(
                class_weight="balanced", y=yf
            )
        pipe.fit(Xf, yf, **fit_kwargs)
        y_prob = pipe.predict_proba(Xv)
        yv_bin = label_binarize(yv, classes=CLASSES)
        scores.append(
            roc_auc_score(yv_bin, y_prob, multi_class="ovr", average="macro")
        )
    return float(np.mean(scores))

In [18]:
# ── 5. Optuna objectives ──────────────────────────────────────────────────────
# n_trials: each trial = one hyperparameter configuration tested via 5-fold CV.
# Optuna uses Bayesian search (TPE) — it learns from previous trials and
# proposes configurations more likely to improve the objective, unlike grid
# or random search which have no memory of past results.
# More trials = better search coverage, linearly more compute time.
# Diminishing returns typically set in after 60-100 trials for complex models.
# XGBoost has 9 hyperparameters so needs more trials than LDA which has 1.

def xgb_objective(trial):
    pipe_xgb.set_params(**{
        "model__n_estimators":     trial.suggest_int("n_estimators", 100, 600),
        "model__max_depth":        trial.suggest_int("max_depth", 3, 10),
        "model__learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "model__subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "model__colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "model__min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "model__gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "model__reg_alpha":        trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "model__reg_lambda":       trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    })
    return cv_macro_auroc(pipe_xgb, X_train, y_train, use_sample_weight=True)

def rf_objective(trial):
    pipe_rf.set_params(**{
        "model__n_estimators":      trial.suggest_int("n_estimators", 100, 600),
        "model__max_depth":         trial.suggest_categorical(
                                        "max_depth", [5, 10, 15, 20, None]),
        "model__max_features":      trial.suggest_float("max_features", 0.05, 0.5),
        "model__min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 20),
        "model__min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
    })
    return cv_macro_auroc(pipe_rf, X_train, y_train, use_sample_weight=False)

def lasso_objective(trial):
    pipe_lasso.set_params(model__C=trial.suggest_float("C", 1e-4, 10.0, log=True))
    return cv_macro_auroc(pipe_lasso, X_train, y_train, use_sample_weight=False)

def lda_objective(trial):
    pipe_lda.set_params(
        model__shrinkage=trial.suggest_float("shrinkage", 0.0, 1.0)
    )
    return cv_macro_auroc(pipe_lda, X_train, y_train, use_sample_weight=False)

In [19]:
# ── 6. Run tuning (all within X_train, X_test never touched) ─────────────────
print("\nTuning XGBoost  (60 trials × 5-fold, est. ~100 min)...")
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_objective, n_trials=60)
print(f"  Best CV AUROC: {study_xgb.best_value:.4f} | params: {study_xgb.best_params}")

print("\nTuning RF       (40 trials × 5-fold, est. ~85 min)...")
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(rf_objective, n_trials=40)
print(f"  Best CV AUROC: {study_rf.best_value:.4f} | params: {study_rf.best_params}")

print("\nTuning LASSO    (20 trials × 5-fold, est. ~61 min)...")
study_lasso = optuna.create_study(direction="maximize")
study_lasso.optimize(lasso_objective, n_trials=20)
print(f"  Best CV AUROC: {study_lasso.best_value:.4f} | C: {study_lasso.best_params['C']:.6f}")

print("\nTuning LDA      (20 trials × 5-fold, est. <1 min)...")
study_lda = optuna.create_study(direction="maximize")
study_lda.optimize(lda_objective, n_trials=20)
print(f"  Best CV AUROC: {study_lda.best_value:.4f} | shrinkage: {study_lda.best_params['shrinkage']:.4f}")


Tuning XGBoost  (60 trials × 5-fold, est. ~100 min)...
  Best CV AUROC: 0.7160 | params: {'n_estimators': 542, 'max_depth': 10, 'learning_rate': 0.010572882850635458, 'subsample': 0.6234365656968217, 'colsample_bytree': 0.4279318940421084, 'min_child_weight': 9, 'gamma': 0.01906712989200654, 'reg_alpha': 0.0003116306538988946, 'reg_lambda': 2.0457498945501547}

Tuning RF       (40 trials × 5-fold, est. ~85 min)...
  Best CV AUROC: 0.7129 | params: {'n_estimators': 596, 'max_depth': None, 'max_features': 0.12427479444448516, 'min_samples_leaf': 8, 'min_samples_split': 4}

Tuning LASSO    (20 trials × 5-fold, est. ~61 min)...


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_

/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_r

/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: Convergen

/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_r

  Best CV AUROC: 0.6865 | C: 0.013881

Tuning LDA      (20 trials × 5-fold, est. <1 min)...
  Best CV AUROC: 0.7044 | shrinkage: 0.0120


In [36]:
# ── 7. Refit on full training set with best hyperparameters ───────────────────
# After tuning selects the best configuration, refit on ALL 80% training data.
# The hyperparameters are fixed — only the amount of training data increases.

print("\nRefitting on full training set...")

# Impute once using full training set as the fitting source.
# X_test receives imputation parameters learned from X_train only.
X_train_imp, X_test_imp = domain_impute(X_train, y_train, X_test)


Refitting on full training set...


In [33]:
# ── Diagnose residual NaN after imputation ────────────────────────────────────
remaining_nan = np.isnan(X_train_imp).sum(axis=0)
nan_cols = [(feature_cols[i], int(remaining_nan[i])) 
            for i in range(len(feature_cols)) if remaining_nan[i] > 0]

print(f"Residual NaN in X_train_imp: {int(remaining_nan.sum())} values")
print(f"Affected features ({len(nan_cols)}):")
for col, n in sorted(nan_cols, key=lambda x: -x[1]):
    in_cohort = i in COHORT_MEDIAN_COLS
    in_zero   = i in ZERO_FILL_COLS
    print(f"  {col}: {n} NaN  | cohort_group={in_cohort} | zero_group={in_zero}")

Residual NaN in X_train_imp: 0 values
Affected features (0):


In [37]:
best_xgb_params = {f"model__{k}": v for k, v in study_xgb.best_params.items()}
pipe_xgb.set_params(**best_xgb_params)
sw_full = compute_sample_weight(class_weight="balanced", y=y_train)
pipe_xgb.fit(X_train_imp, y_train, model__sample_weight=sw_full)

Pipeline(steps=[('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.4279318940421084, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='mlogloss',
                               feature_types=None, feature_weights=None,
                               gamma=0.01906712989200654, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.010572882850635458, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=10,
                               max_leaves=None, min_child_weight=9, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=542, n_jobs=-1, num_class=4, ...))])

In [38]:
best_rf_params = {f"model__{k}": v for k, v in study_rf.best_params.items()}
pipe_rf.set_params(**best_rf_params)
pipe_rf.fit(X_train_imp, y_train)

Pipeline(steps=[('model',
                 RandomForestClassifier(class_weight='balanced',
                                        max_features=0.12427479444448516,
                                        min_samples_leaf=8, min_samples_split=4,
                                        n_estimators=596, n_jobs=-1,
                                        random_state=42))])

In [39]:
pipe_lasso.set_params(model__C=study_lasso.best_params["C"])
pipe_lasso.fit(X_train_imp, y_train)

/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Pipeline(steps=[('scale', RobustScaler()),
                ('model',
                 LogisticRegression(C=0.01388073509520369,
                                    class_weight='balanced', l1_ratio=1.0,
                                    max_iter=2000, random_state=42,
                                    solver='saga'))])

In [40]:
pipe_lda.set_params(model__shrinkage=study_lda.best_params["shrinkage"])
pipe_lda.fit(X_train_imp, y_train)

Pipeline(steps=[('scale', RobustScaler()),
                ('model',
                 LinearDiscriminantAnalysis(shrinkage=0.011969588114692425,
                                            solver='lsqr'))])

In [41]:
import joblib
# ── Save fitted models ────────────────────────────────────────────────────────
# Each file contains the full pipeline (scaler if applicable + model).
# To reload and predict on new data:
#   pipe = joblib.load("results/model_XGBoost.joblib")
#   y_prob = pipe.predict_proba(X_new_imputed)
joblib.dump(pipe_xgb,   "/home/jupyter/workspaces/result/final/classifier/model_XGBoost.joblib")
joblib.dump(pipe_rf,    "/home/jupyter/workspaces/result/final/classifier/model_RandomForest.joblib")

['/home/jupyter/workspaces/result/final/classifier/model_RandomForest.joblib']

In [42]:
joblib.dump(pipe_lasso, "/home/jupyter/workspaces/result/final/classifier/model_LASSO.joblib")
joblib.dump(pipe_lda,   "/home/jupyter/workspaces/result/final/classifier/model_LDA.joblib")

# import joblib
# pipe = joblib.load("/home/jupyter/workspaces/result/final/classifier/model_XGBoost.joblib")
# y_prob = pipe.predict_proba(X_new_imputed)  # X_new must already be imputed

['/home/jupyter/workspaces/result/final/classifier/model_LDA.joblib']

In [45]:
# ── 8. Evaluation — test set used exactly once ────────────────────────────────
def evaluate(name, pipe, X_tr, y_tr, X_te, y_te, best_params, cv_auroc):
    y_tr_bin = label_binarize(y_tr, classes=CLASSES)
    y_te_bin = label_binarize(y_te, classes=CLASSES)

    # Training metrics — large gap vs test indicates overfitting
    tr_prob      = pipe.predict_proba(X_tr)
    train_auroc  = roc_auc_score(y_tr_bin, tr_prob, multi_class="ovr", average="macro")

    # Test metrics
    te_prob     = pipe.predict_proba(X_te)
    te_pred     = pipe.predict(X_te)
    test_auroc  = roc_auc_score(y_te_bin, te_prob, multi_class="ovr", average="macro")
    macro_f1    = f1_score(y_te, te_pred, average="macro")
    weighted_f1 = f1_score(y_te, te_pred, average="weighted")
    brier       = float(np.mean([
        brier_score_loss(y_te_bin[:, i], te_prob[:, i]) for i in range(4)
    ]))
    cm          = confusion_matrix(y_te, te_pred)
    report_dict = classification_report(
        y_te, te_pred, target_names=COHORT_NAMES, output_dict=True, zero_division=0
    )
    per_class_auroc = {
        COHORT_NAMES[i]: float(roc_auc_score(y_te_bin[:, i], te_prob[:, i]))
        for i in range(4)
    }

    gap  = train_auroc - test_auroc
    flag = "  *** possible overfit — review regularization" if gap > 0.05 else ""

    print(f"\n{'='*58}")
    print(f"  {name}")
    print(f"{'='*58}")
    print(f"  CV    macro-AUROC : {cv_auroc:.4f}")
    print(f"  Train macro-AUROC : {train_auroc:.4f}{flag}")
    print(f"  Test  macro-AUROC : {test_auroc:.4f}  (train-test gap: {gap:+.4f})")
    print(f"  Macro F1          : {macro_f1:.4f}")
    print(f"  Weighted F1       : {weighted_f1:.4f}")
    print(f"  Mean Brier score  : {brier:.4f}")
    print(f"\n  Per-class test AUROC:")
    for c, v in per_class_auroc.items():
        print(f"    {c:12s}: {v:.4f}")
    print(f"\n  Classification report:")
    print(classification_report(y_te, te_pred, target_names=COHORT_NAMES, zero_division=0))
    print(f"  Confusion matrix (rows=true, cols=predicted):")
    cm_df = pd.DataFrame(cm, index=COHORT_NAMES, columns=COHORT_NAMES)
    print(cm_df.to_string())

    # Confusion matrix heatmap
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=COHORT_NAMES,
        yticklabels=COHORT_NAMES,
        ax=ax
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"Confusion matrix — {name}")
    plt.tight_layout()
    fig.savefig(f"/home/jupyter/workspaces/result/final/classifier/confusion_matrix_{name}.png", dpi=150)
    plt.close(fig)

    return {
        "model":             name,
        "timestamp":         datetime.now().isoformat(),
        "n_train":           int(len(y_tr)),
        "n_test":            int(len(y_te)),
        "cv_macro_auroc":    round(float(cv_auroc),    4),
        "train_macro_auroc": round(float(train_auroc), 4),
        "test_macro_auroc":  round(float(test_auroc),  4),
        "train_test_gap":    round(float(gap),         4),
        "macro_f1":          round(float(macro_f1),    4),
        "weighted_f1":       round(float(weighted_f1), 4),
        "mean_brier_score":  round(float(brier),       4),
        "per_class_auroc":   {k: round(v, 4) for k, v in per_class_auroc.items()},
        "per_class_report":  report_dict,
        "confusion_matrix":  cm.tolist(),
        "best_params":       best_params,
    }

all_results = []
for model_name, pipe, best_p, cv_auc in [
    ("XGBoost",      pipe_xgb,   study_xgb.best_params,   study_xgb.best_value),
    ("RandomForest", pipe_rf,    study_rf.best_params,    study_rf.best_value),
    ("LASSO",        pipe_lasso, {"C": study_lasso.best_params["C"]}, study_lasso.best_value),
    ("LDA",          pipe_lda,   study_lda.best_params,   study_lda.best_value),
]:
    res = evaluate(model_name, pipe, X_train_imp, y_train, X_test_imp, y_test, best_p, cv_auc)
    all_results.append(res)


  XGBoost
  CV    macro-AUROC : 0.7160
  Train macro-AUROC : 0.9992  *** possible overfit — review regularization
  Test  macro-AUROC : 0.7035  (train-test gap: +0.2957)
  Macro F1          : 0.3228
  Weighted F1       : 0.6370
  Mean Brier score  : 0.1232

  Per-class test AUROC:
    Healthy     : 0.6409
    Insomnia    : 0.6136
    Diabetes    : 0.6929
    Prostate    : 0.8664

  Classification report:
              precision    recall  f1-score   support

     Healthy       0.78      0.79      0.79      1277
    Insomnia       0.21      0.13      0.16       209
    Diabetes       0.22      0.26      0.24       191
    Prostate       0.08      0.16      0.10        32

    accuracy                           0.64      1709
   macro avg       0.32      0.33      0.32      1709
weighted avg       0.64      0.64      0.64      1709

  Confusion matrix (rows=true, cols=predicted):
          Healthy  Insomnia  Diabetes  Prostate
Healthy      1011        77       145        44
Insomnia    

In [46]:
# ── 9. Comparison plots ───────────────────────────────────────────────────────
models       = [r["model"] for r in all_results]
cv_scores    = [r["cv_macro_auroc"]    for r in all_results]
train_scores = [r["train_macro_auroc"] for r in all_results]
test_scores  = [r["test_macro_auroc"]  for r in all_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(models))
w = 0.25
axes[0].bar(x - w, train_scores, width=w, label="Train AUROC", color="#5b9bd5")
axes[0].bar(x,     cv_scores,    width=w, label="CV AUROC",    color="#ed7d31")
axes[0].bar(x + w, test_scores,  width=w, label="Test AUROC",  color="#70ad47")
axes[0].set_xticks(x); axes[0].set_xticklabels(models)
axes[0].set_ylim(0.5, 1.01)
axes[0].set_ylabel("Macro AUROC (one-vs-rest)")
axes[0].set_title("Train / CV / Test AUROC by model")
axes[0].axhline(0.5, color="grey", linestyle="--", linewidth=0.8, label="chance")
axes[0].legend()  # called after axhline so 'chance' appears in legend

colors = ["#5b9bd5", "#ed7d31", "#70ad47", "#ffc000"]
x2 = np.arange(len(COHORT_NAMES))
for i, (mname, col) in enumerate(zip(models, colors)):
    vals = [all_results[i]["per_class_auroc"][c] for c in COHORT_NAMES]
    axes[1].bar(x2 + (i - 1.5) * 0.2, vals, width=0.2, label=mname, color=col)
axes[1].set_xticks(x2); axes[1].set_xticklabels(COHORT_NAMES)
axes[1].set_ylim(0.5, 1.01)
axes[1].set_ylabel("AUROC")
axes[1].set_title("Per-class test AUROC by model")
axes[1].axhline(0.5, color="grey", linestyle="--", linewidth=0.8, label="chance")
axes[1].legend()  # called after axhline so 'chance' appears in legend

plt.tight_layout()
fig.savefig("/home/jupyter/workspaces/result/final/classifier/model_comparison.png", dpi=150)
plt.close(fig)

In [47]:
# ── 10. Save results ──────────────────────────────────────────────────────────
# JSON: full detail, machine-readable
with open("/home/jupyter/workspaces/result/final/classifier/all_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

# TXT: human-readable report
with open("/home/jupyter/workspaces/result/final/classifier/summary.txt", "w") as f:
    f.write(f"Fitbit 4-cohort classification results\n")
    f.write(f"Run: {datetime.now().isoformat()}\n")
    f.write(f"Train N={len(y_train)} | Test N={len(y_test)}\n")
    f.write("="*58 + "\n\n")
    for r in all_results:
        f.write(f"Model             : {r['model']}\n")
        f.write(f"CV    macro-AUROC : {r['cv_macro_auroc']}\n")
        f.write(f"Train macro-AUROC : {r['train_macro_auroc']}\n")
        f.write(f"Test  macro-AUROC : {r['test_macro_auroc']}  "
                f"(gap: {r['train_test_gap']:+.4f})\n")
        f.write(f"Macro F1          : {r['macro_f1']}\n")
        f.write(f"Weighted F1       : {r['weighted_f1']}\n")
        f.write(f"Mean Brier score  : {r['mean_brier_score']}\n")
        f.write(f"Per-class AUROC   : {r['per_class_auroc']}\n")
        f.write(f"Best params       : {r['best_params']}\n")
        f.write(f"Confusion matrix  (rows=true, cols=predicted):\n")
        cm_df = pd.DataFrame(
            r["confusion_matrix"], index=COHORT_NAMES, columns=COHORT_NAMES
        )
        f.write(cm_df.to_string() + "\n\n")

In [48]:
# CSV: one row per model, opens directly in Excel / pandas
summary_rows = []
for r in all_results:
    row = {
        "model":             r["model"],
        "cv_macro_auroc":    r["cv_macro_auroc"],
        "train_macro_auroc": r["train_macro_auroc"],
        "test_macro_auroc":  r["test_macro_auroc"],
        "train_test_gap":    r["train_test_gap"],
        "macro_f1":          r["macro_f1"],
        "weighted_f1":       r["weighted_f1"],
        "mean_brier_score":  r["mean_brier_score"],
    }
    row.update({f"auroc_{k}": v for k, v in r["per_class_auroc"].items()})
    summary_rows.append(row)
pd.DataFrame(summary_rows).to_csv("/home/jupyter/workspaces/result/final/classifier/summary.csv", index=False)

print("\n" + "="*58)
print("Saved to /home/jupyter/workspaces/result/final/classifier/")
print("  all_results.json           full detail for future analysis")
print("  summary.txt                human-readable report")
print("  summary.csv                table for Excel / pandas")
print("  confusion_matrix_<m>.png   per-model heatmaps")
print("  model_comparison.png       AUROC bar chart")


Saved to results/
  all_results.json           full detail for future analysis
  summary.txt                human-readable report
  summary.csv                table for Excel / pandas
  confusion_matrix_<m>.png   per-model heatmaps
  model_comparison.png       AUROC bar chart


In [69]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def generate_model_reports(pipelines, feature_names, save_path="/home/jupyter/workspaces/result/final/classifier/"):
    out_dir = Path(save_path)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    sns.set_theme(style="whitegrid")
    TOP_N = 30

    for model_name, pipe in pipelines.items():
        try:
            model = pipe.named_steps['model']
            
            # 2. Extract Scores logic
            if hasattr(model, 'feature_importances_'):
                scores = model.feature_importances_
            elif hasattr(model, 'coef_'):
                # Handle multi-class (4 classes) by taking mean of absolute values
                scores = np.abs(model.coef_).mean(axis=0)
            else:
                print(f"Skipping {model_name}: No importance attribute.")
                continue

            # 3. Create DataFrame and Sort
            # Using the feature_names passed into the function
            df = pd.DataFrame({'Feature': feature_names, 'Importance': scores})
            df = df.sort_values(by='Importance', ascending=False).head(TOP_N)
            print(df)
            # 4. Visualization (Fixed for compatibility)
            plt.figure(figsize=(12, 8))
            
            # Use hue to avoid palette warnings, but handle legend separately
            ax = sns.barplot(
                x='Importance', 
                y='Feature', 
                data=df
                #hue='Feature' 
                #palette='viridis'
            )
            
            # Remove legend if it exists (avoids the 'legend' keyword bug)
            if ax.get_legend() is not None:
                ax.get_legend().remove()

            #plt.title(f'Top {TOP_N} Features: {model_name}', fontsize=15, fontweight='bold')
            plt.xlabel('Importance Score')
            plt.ylabel('')
            plt.tight_layout()

            # 5. Local Saving logic
            safe_name = model_name.lower().replace(' ', '_')
            full_filename = str(out_dir / f"{safe_name}_importance.png")
            
            plt.savefig(full_filename, dpi=300, bbox_inches='tight')
            print(f"Successfully saved: {full_filename}")
            
            plt.show()
            plt.close() # Critical for memory

        except Exception as e:
            print(f"Error processing {model_name}: {e}")

In [ ]:
fitted_pipelines = {
    "XGBoost": pipe_xgb,
    "Random Forest": pipe_rf,
    "LASSO": pipe_lasso,
    "LDA": pipe_lda
}

# Run the fixed pipeline
generate_model_reports(
    pipelines=fitted_pipelines, 
    feature_names=feature_cols,
    save_path="/home/jupyter/workspaces/result/final/classifier/"
)

                                   Feature  Importance
131          hr_cardio_min_heart_rate_mean    0.019414
124        hr_fat_burn_max_heart_rate_mean    0.018565
8                      calories_out_median    0.018124
135           hr_cardio_min_heart_rate_p90    0.015832
149             hr_peak_min_heart_rate_p90    0.015706
7                        calories_out_mean    0.015415
128         hr_fat_burn_max_heart_rate_p90    0.013613
125      hr_fat_burn_max_heart_rate_median    0.012734
132        hr_cardio_min_heart_rate_median    0.012601
118      hr_fat_burn_min_heart_rate_median    0.012578
121         hr_fat_burn_min_heart_rate_p90    0.012350
146          hr_peak_min_heart_rate_median    0.012350
138          hr_cardio_max_heart_rate_mean    0.011790
114     hr_out_of_range_max_heart_rate_p90    0.011756
142           hr_cardio_max_heart_rate_p90    0.011718
145            hr_peak_min_heart_rate_mean    0.011678
11                        calories_out_p90    0.010805
139       

In [ ]:
# 2. Extract Importance from Pipelines
feature_names = X_train_imp.columns

# --- XGBoost ---
xgb_scores = pipe_xgb.named_steps['model'].feature_importances_
plot_individual_importance(feature_names, xgb_scores, "XGBoost", "viridis")

# --- Random Forest ---
rf_scores = pipe_rf.named_steps['model'].feature_importances_
plot_individual_importance(feature_names, rf_scores, "Random Forest", "magma")

# --- LASSO (Logistic Regression) ---
# Multi-class coef_ is (n_classes, n_features). 
# Taking the mean of absolute values across classes provides a global importance metric.
lasso_coefs = np.abs(pipe_lasso.named_steps['model'].coef_).mean(axis=0)
plot_individual_importance(feature_names, lasso_coefs, "LASSO (L1 Coefficients)", "rocket")

# --- LDA ---
# Similarly, LDA coef_ is (n_classes - 1, n_features) or (n_classes, n_features)
lda_coefs = np.abs(pipe_lda.named_steps['model'].coef_).mean(axis=0)
plot_individual_importance(feature_names, lda_coefs, "LDA (Coefficients)", "mako")

In [21]:
features.to_csv("/home/jupyter/workspaces/result/final/classifier/fitbit_features.txt", sep="\t", index=False)

In [26]:
len(activity['person_id'].unique())

8926

In [30]:
print(features.columns.tolist()[:30])

['person_id', 'activity_calories_mean', 'activity_calories_median', 'activity_calories_sd', 'activity_calories_p10', 'activity_calories_p90', 'activity_calories_slope', 'activity_calories_mad', 'calories_bmr_mean', 'calories_bmr_median', 'calories_bmr_sd', 'calories_bmr_p10', 'calories_bmr_p90', 'calories_bmr_slope', 'calories_bmr_mad', 'calories_out_mean', 'calories_out_median', 'calories_out_sd', 'calories_out_p10', 'calories_out_p90', 'calories_out_slope', 'calories_out_mad', 'elevation_mean', 'elevation_median', 'elevation_sd', 'elevation_p10', 'elevation_p90', 'elevation_slope', 'elevation_mad', 'fairly_active_minutes_mean']
